In [1]:
try:
    import yfinance
except:
    %pip pip install yfinance

In [2]:
# ============================================================
# 01_us_stock_download.ipynb
# 미국 주식 상승추세 스크리너 - 데이터 수집
# ============================================================

# 처음 한 번만 실행
# %pip install -U yfinance pandas


# --------------------------------------------------
# 1. 라이브러리
# --------------------------------------------------
from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import time

import pandas as pd
import yfinance as yf
from IPython.display import display


# --------------------------------------------------
# 2. 설정
# --------------------------------------------------
START_DATE = "2015-01-01"
BATCH_SIZE = 100

# 배당/분할로 과거 조정가격이 바뀔 수 있으므로 최근 400일 재다운로드
REFRESH_DAYS = 400

BASE_DIR = Path("us_stock_data")
STOCK_DIR = BASE_DIR / "stocks"
MARKET_DIR = BASE_DIR / "market"
META_DIR = BASE_DIR / "metadata"
HISTORY_DIR = META_DIR / "universe_history"

for folder in [STOCK_DIR, MARKET_DIR, META_DIR, HISTORY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MARKET_TICKERS = ["SPY", "QQQ"]

print("데이터 저장 폴더 :", BASE_DIR.resolve())


# --------------------------------------------------
# 3. S&P 500 종목 목록
# --------------------------------------------------
SP500_URL = (
    "https://raw.githubusercontent.com/"
    "datasets/s-and-p-500-companies/main/data/constituents.csv"
)

SP500_FILE = META_DIR / "sp500_constituents.csv"

try:
    print("\nS&P 500 종목 목록 다운로드 중...")
    sp500 = pd.read_csv(SP500_URL)
    print("S&P 500 목록 다운로드 성공")

except Exception as e:
    print("S&P 500 목록 다운로드 실패 :", e)

    if SP500_FILE.exists():
        print("기존 저장 파일을 사용합니다.")
        sp500 = pd.read_csv(SP500_FILE)
    else:
        raise RuntimeError("S&P 500 목록 다운로드 실패 + 기존 CSV 없음")


# 컬럼 이름 정리
sp500 = sp500.rename(columns={
    "Symbol": "Ticker",
    "Security": "Name",
    "GICS Sector": "Sector",
    "GICS Sub-Industry": "Industry"
})

# Yahoo Finance 티커 형식: BRK.B → BRK-B
sp500["YahooTicker"] = sp500["Ticker"].astype(str).str.replace(".", "-", regex=False)

sp500 = sp500[
    ["Ticker", "YahooTicker", "Name", "Sector", "Industry"]
].sort_values("Ticker").reset_index(drop=True)

# 최신 S&P 500 목록 저장
sp500.to_csv(SP500_FILE, index=False)

print("S&P 500 증권 수 :", len(sp500))
display(sp500)


# --------------------------------------------------
# 4. S&P 500 Universe 스냅샷
# --------------------------------------------------
# 앞으로 쌓이는 스냅샷은 백테스트 시 생존편향 완화에 활용
today_ny = datetime.now(ZoneInfo("America/New_York")).date()
snapshot_file = HISTORY_DIR / f"sp500_{today_ny}.csv"

if not snapshot_file.exists():
    sp500.to_csv(snapshot_file, index=False)
    print("\nUniverse 스냅샷 저장 :", snapshot_file)
else:
    print("\n오늘 Universe 스냅샷 존재 :", snapshot_file)


# --------------------------------------------------
# 5. 최신 완료 거래일 확인
# --------------------------------------------------
now_ny = datetime.now(ZoneInfo("America/New_York"))

# 미국 동부시간 18시 전에는 오늘 일봉을 사용하지 않음
if now_ny.hour >= 18:
    download_end = now_ny.date() + timedelta(days=1)
else:
    download_end = now_ny.date()

DOWNLOAD_END = download_end.isoformat()
spy_start = (now_ny.date() - timedelta(days=14)).isoformat()

# SPY로 실제 최신 거래일 확인
spy_check = yf.download(
    "SPY",
    start=spy_start,
    end=DOWNLOAD_END,
    interval="1d",
    auto_adjust=True,
    progress=False,
    threads=False,
    multi_level_index=False
)

if spy_check.empty:
    raise RuntimeError("SPY 데이터를 다운로드하지 못했습니다.")

LATEST_MARKET_DATE = pd.Timestamp(spy_check.index[-1]).date()

print("\n현재 미국 동부시간 :", now_ny.strftime("%Y-%m-%d %H:%M"))
print("최신 완료 거래일   :", LATEST_MARKET_DATE)


# --------------------------------------------------
# 6. 다운로드 대상 준비
# --------------------------------------------------
targets = []

# S&P 500
for ticker in sp500["YahooTicker"]:
    targets.append({
        "Ticker": ticker,
        "Type": "Stock",
        "Path": STOCK_DIR / f"{ticker}.csv"
    })

# SPY / QQQ
for ticker in MARKET_TICKERS:
    targets.append({
        "Ticker": ticker,
        "Type": "Market",
        "Path": MARKET_DIR / f"{ticker}.csv"
    })

print("\nS&P 500 종목 :", len(sp500))
print("시장 ETF     :", len(MARKET_TICKERS))
print("전체 대상    :", len(targets))


# --------------------------------------------------
# 7. 갱신이 필요한 종목 찾기
# --------------------------------------------------
pending = []
latest_count = 0

for item in targets:

    ticker = item["Ticker"]
    path = item["Path"]
    last_date = None

    # 기존 CSV의 마지막 날짜 확인
    if path.exists():
        try:
            dates = pd.read_csv(path, usecols=["Date"])

            if not dates.empty:
                last_date = pd.to_datetime(dates["Date"]).max().date()

        except Exception as e:
            print(ticker, "기존 CSV 읽기 오류 :", e)

    # 이미 최신이면 건너뜀
    if last_date is not None and last_date >= LATEST_MARKET_DATE:
        latest_count += 1
        continue

    # 처음 다운로드
    if last_date is None:
        start_date = START_DATE

    # 기존 파일 갱신: 최근 400일 다시 다운로드
    else:
        start_date = pd.Timestamp(last_date) - pd.Timedelta(days=REFRESH_DAYS)
        start_date = max(start_date, pd.Timestamp(START_DATE))
        start_date = start_date.date().isoformat()

    pending.append({
        "Ticker": ticker,
        "Type": item["Type"],
        "Path": path,
        "StartDate": start_date
    })

print("\n전체 대상 :", len(targets))
print("이미 최신 :", latest_count)
print("갱신 필요 :", len(pending))


# --------------------------------------------------
# 8. OHLCV 다운로드
# --------------------------------------------------
failed = []

if len(pending) == 0:
    print("\n모든 데이터가 최신입니다.")

else:
    pending_df = pd.DataFrame(pending)

    # 같은 시작일을 가진 종목끼리 다운로드
    for start_date, group in pending_df.groupby("StartDate"):

        group = group.reset_index(drop=True)

        # BATCH_SIZE개씩 다운로드
        for i in range(0, len(group), BATCH_SIZE):

            batch = group.iloc[i:i + BATCH_SIZE]
            tickers = batch["Ticker"].tolist()

            print(
                f"\n다운로드 {i + 1} ~ {min(i + BATCH_SIZE, len(group))}"
                f" / {len(group)} | 시작일 {start_date}"
            )

            try:
                data = yf.download(
                    tickers,
                    start=start_date,
                    end=DOWNLOAD_END,
                    interval="1d",
                    auto_adjust=True,
                    actions=False,
                    group_by="ticker",
                    progress=False,
                    threads=True,
                    multi_level_index=True
                )

            except Exception as e:
                print("배치 다운로드 오류 :", e)
                failed.extend(tickers)
                continue

            # --------------------------------------------------
            # 종목별 CSV 저장
            # --------------------------------------------------
            for _, row in batch.iterrows():

                ticker = row["Ticker"]
                path = row["Path"]

                try:
                    if data.empty:
                        failed.append(ticker)
                        continue

                    # yfinance MultiIndex에서 종목 데이터 추출
                    if isinstance(data.columns, pd.MultiIndex):

                        level0 = data.columns.get_level_values(0)
                        level1 = data.columns.get_level_values(1)

                        if ticker in level0:
                            df = data[ticker].copy()

                        elif ticker in level1:
                            df = data.xs(ticker, axis=1, level=1).copy()

                        else:
                            failed.append(ticker)
                            continue

                    else:
                        df = data.copy()

                    # Index를 Date 컬럼으로 변경
                    df = df.reset_index()
                    df = df.rename(columns={df.columns[0]: "Date"})

                    # 필요한 OHLCV만 사용
                    keep_cols = ["Date", "Open", "High", "Low", "Close", "Volume"]
                    df = df[[col for col in keep_cols if col in df.columns]]

                    if "Close" not in df.columns:
                        failed.append(ticker)
                        continue

                    df["Date"] = pd.to_datetime(df["Date"])
                    df = df.dropna(subset=["Close"])

                    if df.empty:
                        failed.append(ticker)
                        continue

                    # 기존 CSV와 새 데이터 병합
                    if path.exists():
                        old = pd.read_csv(path, parse_dates=["Date"])
                        df = pd.concat([old, df], ignore_index=True)

                    # 같은 날짜는 새로 받은 값 사용
                    df = (
                        df.drop_duplicates(subset="Date", keep="last")
                        .sort_values("Date")
                        .reset_index(drop=True)
                    )

                    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")
                    df.to_csv(path, index=False)

                except Exception as e:
                    print(ticker, "저장 오류 :", e)
                    failed.append(ticker)

            # Yahoo Finance에 너무 빠르게 반복 요청하지 않도록 잠시 대기
            time.sleep(1)


# --------------------------------------------------
# 9. 실패 종목 확인
# --------------------------------------------------
failed = sorted(set(failed))

print("\n" + "=" * 60)
print("다운로드 완료")
print("=" * 60)

if failed:
    print("실패 종목 :", len(failed))
    print(failed[:30])
else:
    print("실패 종목 없음")


# --------------------------------------------------
# 10. 저장 결과 검증
# --------------------------------------------------
summary = []

for item in targets:

    ticker = item["Ticker"]
    path = item["Path"]

    rows = 0
    last_date = None
    status = "파일없음"

    if path.exists():
        try:
            temp = pd.read_csv(path, usecols=["Date"])
            rows = len(temp)

            if rows > 0:
                last_date = pd.to_datetime(temp["Date"]).max().date()
                status = "OK" if last_date >= LATEST_MARKET_DATE else "미완료"
            else:
                status = "오류"

        except Exception:
            status = "오류"

    summary.append({
        "Ticker": ticker,
        "Type": item["Type"],
        "Rows": rows,
        "LastDate": last_date,
        "Status": status
    })

summary = pd.DataFrame(summary)

print("\n상태별 개수")
display(summary["Status"].value_counts().to_frame("종목수"))

print("\n시장 데이터")
display(summary[summary["Type"] == "Market"])

problem = summary[summary["Status"] != "OK"]

print("\n문제가 있는 종목 :", len(problem))
display(problem.head(30))


# --------------------------------------------------
# 11. 저장 데이터 확인
# --------------------------------------------------
sample_ticker = "AAPL"
sample_file = STOCK_DIR / f"{sample_ticker}.csv"

if sample_file.exists():
    sample = pd.read_csv(sample_file)

    print(f"\n{sample_ticker} 최근 데이터")
    display(sample)


# --------------------------------------------------
# 12. Sector별 종목 수
# --------------------------------------------------
print("\nSector별 종목 수")
display(sp500["Sector"].value_counts().to_frame("종목수"))

데이터 저장 폴더 : /Users/back/내 드라이브/_2026/02_2026_03/02_한화시스템_강의/_git_ds1/work/_us_stock/us_stock_data

S&P 500 종목 목록 다운로드 중...
S&P 500 목록 다운로드 성공
S&P 500 증권 수 : 503


,Ticker,YahooTicker,Name,Sector,Industry
0,A,A,Agilent Technologies,Health Care,Life Sciences Tools & Services
1,AAPL,AAPL,Apple Inc.,Information Technology,"Technology Hardware, Storage & Peripherals"
2,ABBV,ABBV,AbbVie,Health Care,Biotechnology
3,ABNB,ABNB,Airbnb,Consumer Discretionary,"Hotels, Resorts & Cruise Lines"
4,ABT,ABT,Abbott Laboratories,Health Care,Health Care Equipment
...,...,...,...,...,...
498,XYZ,XYZ,"Block, Inc.",Financials,Transaction & Payment Processing Services
499,YUM,YUM,Yum! Brands,Consumer Discretionary,Restaurants
500,ZBH,ZBH,Zimmer Biomet,Health Care,Health Care Equipment
501,ZBRA,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments



오늘 Universe 스냅샷 존재 : us_stock_data/metadata/universe_history/sp500_2026-08-14.csv

현재 미국 동부시간 : 2026-08-14 22:08
최신 완료 거래일   : 2026-08-14

S&P 500 종목 : 503
시장 ETF     : 2
전체 대상    : 505

전체 대상 : 505
이미 최신 : 505
갱신 필요 : 0

모든 데이터가 최신입니다.

다운로드 완료
실패 종목 없음

상태별 개수


,종목수
Status,
OK,505



시장 데이터


...

,Ticker,Type,Rows,LastDate,Status



AAPL 최근 데이터


,Date,Open,High,Low,Close,Volume
0,2015-01-02,24.627205,24.638260,23.734002,24.171761,212818400
1,2015-01-05,23.941824,24.021417,23.305086,23.490801,257142000
2,2015-01-06,23.554914,23.751684,23.132632,23.493010,263188400
3,2015-01-07,23.700837,23.921927,23.590292,23.822437,160423600
4,2015-01-08,24.149649,24.795231,24.032470,24.737747,237458000
...,...,...,...,...,...,...
2916,2026-08-10,306.829987,308.260010,304.609985,308.260010,44812500
2917,2026-08-11,307.750000,309.970001,302.790009,304.910004,37476700
2918,2026-08-12,305.100006,305.660004,300.570007,302.250000,41657800
2919,2026-08-13,304.209991,306.000000,302.049988,305.260010,40349300



Sector별 종목 수


,종목수
Sector,
Industrials,83
Financials,76
Information Technology,73
Health Care,59
Consumer Discretionary,47
Consumer Staples,34
Utilities,31
Real Estate,31
Materials,25


In [3]:
# end